# Triton basics: vector add and fused softmax on a T4

**Runtime → Change runtime type → T4 GPU**, then Run all (about 3–5 minutes; the first call of each kernel compiles it).

What you will measure:
1. Tutorial 01 **vector add** in Triton vs `x + y` in PyTorch: GB/s = 3 · numel · 4 bytes / time, next to the T4's 320 GB/s peak.
2. Tutorial 02 **fused softmax** (one program per row, persistent grid) vs `torch.softmax`, the unfused `naive_softmax` (5 PyTorch kernels) and `torch.compile(naive_softmax)`, at M = 4,096 rows.
3. The card's worked example on your T4: the naive vs fused byte count predicts a ~4× gap.

Timing uses `triton.testing.do_bench`. Its `warmup=25` and `rep=100` are **milliseconds of running time, not iteration counts**, and before each timed run it zeroes a 256 MB buffer to flush the L2 cache.

The kernels are adapted from the Triton tutorials `01-vector-add.py` and `02-fused-softmax.py` (https://github.com/triton-lang/triton/tree/main/python/tutorials), Copyright 2018-2020 Philippe Tillet, Copyright 2020-2022 OpenAI, MIT license (https://github.com/triton-lang/triton/blob/main/LICENSE). Changes: the AMD branches are removed, a fallback is added for the occupancy query, and the prints are ours.

This notebook has **not yet been run by the card's author** (no NVIDIA GPU). Your T4 numbers are the exercise.

In [ ]:
!nvidia-smi

In [ ]:
import os, math, torch, triton
import triton.language as tl
print("torch", torch.__version__, "| triton", triton.__version__)
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
DEVICE = torch.device("cuda")
props = torch.cuda.get_device_properties(0)
print(props.name, "|", props.multi_processor_count, "SMs |", round(props.total_memory / 1e9, 1), "GB")
PEAK_GBS = 320.0   # T4, Turing whitepaper (the T4 datasheet prints 300)

## 1. Vector add (tutorial 01)
Each **program** handles one block of `BLOCK_SIZE` elements. There is no thread index anywhere: `tl.arange` gives the whole block of offsets at once, and the compiler splits it across the threads of the program's warps.

In [ ]:
@triton.jit
def add_kernel(x_ptr, y_ptr, output_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(axis=0)                    # which block am I?
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)  # a vector of BLOCK_SIZE indices
    mask = offsets < n_elements                    # guard the ragged last block
    x = tl.load(x_ptr + offsets, mask=mask)
    y = tl.load(y_ptr + offsets, mask=mask)
    tl.store(output_ptr + offsets, x + y, mask=mask)

def add(x, y):
    output = torch.empty_like(x)
    n = output.numel()
    grid = lambda meta: (triton.cdiv(n, meta["BLOCK_SIZE"]),)
    add_kernel[grid](x, y, output, n, BLOCK_SIZE=1024)
    return output        # launch is asynchronous: the kernel may still be running here

torch.manual_seed(0)
size = 98432
x = torch.rand(size, device=DEVICE); y = torch.rand(size, device=DEVICE)
out = add(x, y)
print("programs launched:", triton.cdiv(size, 1024), "| valid lanes in the last one:", size - (triton.cdiv(size, 1024) - 1) * 1024)
print("max |torch - triton| =", torch.max(torch.abs((x + y) - out)).item())
assert torch.allclose(x + y, out)

In [ ]:
print(f"{'size':>12} {'Triton GB/s':>12} {'Torch GB/s':>11} {'Triton % of 320':>16}")
for p in range(12, 28):
    n = 2 ** p
    x = torch.rand(n, device=DEVICE); y = torch.rand(n, device=DEVICE)
    gbps = lambda ms: 3 * n * 4 * 1e-9 / (ms * 1e-3)
    t_tr = triton.testing.do_bench(lambda: add(x, y), return_mode="median")
    t_pt = triton.testing.do_bench(lambda: x + y, return_mode="median")
    print(f"{n:>12,} {gbps(t_tr):>12.1f} {gbps(t_pt):>11.1f} {100 * gbps(t_tr) / PEAK_GBS:>15.1f}%")
    del x, y
# Small sizes: the time is launch overhead, so GB/s rises in proportion to size.
# Large sizes: both should flatten near the same ceiling, below 320 GB/s.

## 2. Fused softmax (tutorial 02)
First the unfused version. The tutorial's comments count element reads and writes per line: in total **read 5MN + 2M, write 3MN + 2M**. The fused kernel reads X once and writes Y once: 2MN.

In [ ]:
def naive_softmax(x):
    x_max = x.max(dim=1)[0]            # read MN,     write M
    z = x - x_max[:, None]             # read MN + M, write MN
    numerator = torch.exp(z)           # read MN,     write MN
    denominator = numerator.sum(dim=1) # read MN,     write M
    ret = numerator / denominator[:, None]  # read MN + M, write MN
    return ret                         # total: read 5MN + 2M, write 3MN + 2M

@triton.jit
def softmax_kernel(output_ptr, input_ptr, input_row_stride, output_row_stride, n_rows, n_cols,
                   BLOCK_SIZE: tl.constexpr, num_stages: tl.constexpr):
    row_start = tl.program_id(0)
    row_step = tl.num_programs(0)      # persistent: each program walks rows strided by the grid size
    for row_idx in tl.range(row_start, n_rows, row_step, num_stages=num_stages):
        row_start_ptr = input_ptr + row_idx * input_row_stride
        col_offsets = tl.arange(0, BLOCK_SIZE)          # BLOCK_SIZE = next power of 2 >= n_cols
        mask = col_offsets < n_cols
        row = tl.load(row_start_ptr + col_offsets, mask=mask, other=-float("inf"))  # padding can't win the max
        row_minus_max = row - tl.max(row, axis=0)       # reduction 1
        numerator = tl.exp(row_minus_max)
        denominator = tl.sum(numerator, axis=0)         # reduction 2
        out = numerator / denominator
        tl.store(output_ptr + row_idx * output_row_stride + col_offsets, out, mask=mask)

In [ ]:
from triton.runtime import driver
dev_props = driver.active.utils.get_device_properties(0)
NUM_SM, NUM_REGS = dev_props["multiprocessor_count"], dev_props["max_num_regs"]
SIZE_SMEM, WARP_SIZE = dev_props["max_shared_mem"], dev_props["warpSize"]
print("SMs", NUM_SM, "| registers per SM", NUM_REGS, "| max shared mem", SIZE_SMEM, "B")
_grid_cache = {}

def softmax(x, verbose=False):
    n_rows, n_cols = x.shape
    BLOCK_SIZE = triton.next_power_of_2(n_cols)
    num_warps = 8
    num_stages = 4 if SIZE_SMEM > 200000 else 2      # the T4 has 64 KB, so 2
    y = torch.empty_like(x)
    key = (BLOCK_SIZE, num_stages)
    if key not in _grid_cache:
        # The tutorial pre-compiles to read register and shared-memory use, then fits
        # as many programs per SM as the register file allows ("occupancy").
        # This compile-introspection API has changed between Triton versions, so fall back if it's missing.
        try:
            k = softmax_kernel.warmup(y, x, x.stride(0), y.stride(0), n_rows, n_cols, BLOCK_SIZE=BLOCK_SIZE,
                                      num_stages=num_stages, num_warps=num_warps, grid=(1,))
            k._init_handles()
            occupancy = NUM_REGS // (k.n_regs * WARP_SIZE * num_warps)
            if k.metadata.shared:
                occupancy = min(occupancy, SIZE_SMEM // k.metadata.shared)
            info = f"n_regs={k.n_regs}, shared={k.metadata.shared} B"
        except Exception as e:
            occupancy, info = 2, f"occupancy query failed ({type(e).__name__}); using 2"
        _grid_cache[key] = max(1, occupancy)
        if verbose:
            print(f"BLOCK_SIZE={BLOCK_SIZE}: {info} -> {_grid_cache[key]} programs/SM x {NUM_SM} SMs")
    num_programs = min(NUM_SM * _grid_cache[key], n_rows)
    softmax_kernel[(num_programs, 1, 1)](y, x, x.stride(0), y.stride(0), n_rows, n_cols, BLOCK_SIZE,
                                         num_stages, num_warps=num_warps)
    return y

torch.manual_seed(0)
x = torch.randn(1823, 781, device=DEVICE)       # irregular shape tests the mask and padding
y_tr = softmax(x, verbose=True)
print("max |torch - triton| =", (y_tr - torch.softmax(x, dim=1)).abs().max().item())
assert torch.allclose(y_tr, torch.softmax(x, dim=1), atol=1e-6)
assert torch.allclose(naive_softmax(x), torch.softmax(x, dim=1), atol=1e-6)

### Benchmark at M = 4,096 rows
The tutorial's GB/s charges **every** provider for 2 · M · N · 4 bytes (the fused minimum), so it is really a speed score. The last column converts the naive line into the bytes it actually moves, (8MN + 4M) · 4, to show it runs near the same memory ceiling while moving ~4× the bytes.

In [ ]:
compiled_naive = torch.compile(naive_softmax)   # TorchInductor generates Triton kernels for this on GPU
M = 4096
print(f"{'N':>6} {'Triton':>8} {'torch.softmax':>14} {'naive':>7} {'compiled':>9}   {'naive actual':>12}  (GB/s; T4 peak 320)")
for N in [256, 1024, 2048, 4096, 8192, 12544]:
    x = torch.randn(M, N, device=DEVICE, dtype=torch.float32)
    gbps = lambda ms: 2 * M * N * 4 * 1e-9 / (ms * 1e-3)
    r = {name: gbps(triton.testing.do_bench(fn, return_mode="median")) for name, fn in [
        ("triton", lambda: softmax(x)), ("torch", lambda: torch.softmax(x, dim=-1)),
        ("naive", lambda: naive_softmax(x)), ("compiled", lambda: compiled_naive(x))]}
    actual = r["naive"] * (8 * M * N + 4 * M) / (2 * M * N)
    print(f"{N:>6} {r['triton']:>8.1f} {r['torch']:>14.1f} {r['naive']:>7.1f} {r['compiled']:>9.1f}   {actual:>12.1f}")
    del x

## 3. The card's worked example on your T4
Softmax over the Llama-3.1-8B vocabulary for 4,096 rows: M = 4,096, N = 128,256, FP32 (2.1 GB of logits). Byte-count floors at 320 GB/s: **naive 16.81 GB → 52.5 ms**, **fused 4.20 GB → 13.1 ms**.

The tutorial kernel is **not** run here: one 128,256-wide row pads to BLOCK_SIZE = 131,072 floats = 512 KiB per program, far more than an SM holds, so it would spill or fail to compile. A vocab-sized softmax needs a kernel that loops over chunks of the row (online softmax, the trick inside FlashAttention). `torch.softmax` and `torch.compile` handle it. Peak memory for the naive version is about 4 × 2.1 GB, which fits in the T4's 15 GB.

In [ ]:
M, N = 4096, 128256
x = torch.randn(M, N, device=DEVICE, dtype=torch.float32)
naive_bytes, fused_bytes = (8 * M * N + 4 * M) * 4, 2 * M * N * 4
print(f"naive bytes {naive_bytes / 1e9:.2f} GB -> floor {naive_bytes / 320e9 * 1e3:.1f} ms | fused bytes {fused_bytes / 1e9:.2f} GB -> floor {fused_bytes / 320e9 * 1e3:.1f} ms")
for name, fn, b in [("naive (5 kernels)", lambda: naive_softmax(x), naive_bytes),
                    ("torch.softmax", lambda: torch.softmax(x, dim=-1), fused_bytes),
                    ("torch.compile(naive)", lambda: compiled_naive(x), fused_bytes)]:
    ms = triton.testing.do_bench(fn, return_mode="median")
    print(f"{name:>22}: {ms:7.2f} ms   {b / (ms * 1e-3) / 1e9:6.1f} GB/s of its own byte count ({100 * b / (ms * 1e-3) / 320e9:.0f}% of 320)")
del x; torch.cuda.empty_cache()

## 4. Interpreter mode: `TRITON_INTERPRET=1`
Triton's README: `TRITON_INTERPRET=1` "uses the Triton interpreter instead of running on the GPU. You can insert Python breakpoints in your kernel code!" It runs the kernel with numpy on the CPU, so you can `print` inside a kernel and step through it. It's slow, so use tiny inputs.

It does **not** get Triton onto a Mac: Triton publishes Linux wheels only (PyPI `triton` 3.8.0 lists manylinux x86_64 and aarch64 wheels, no macOS), so this runs on Colab or Linux. The variable must be set before `triton` is imported, which is why the cell below starts a fresh Python process.

In [ ]:
%%writefile interp_add.py
import os
os.environ["TRITON_INTERPRET"] = "1"      # must be set before importing triton
import torch, triton, triton.language as tl

@triton.jit
def add_kernel(x_ptr, y_ptr, out_ptr, n, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offs < n
    print("program", pid, "offsets", offs, "mask", mask)   # plain Python print works in the interpreter
    tl.store(out_ptr + offs, tl.load(x_ptr + offs, mask=mask) + tl.load(y_ptr + offs, mask=mask), mask=mask)

x = torch.arange(10, dtype=torch.float32)   # CPU tensors: nothing touches the GPU
y = torch.ones(10)
out = torch.empty_like(x)
add_kernel[(triton.cdiv(10, 8),)](x, y, out, 10, BLOCK_SIZE=8)
print(out)

In [ ]:
!python interp_add.py

## Reference numbers and what to compare
The Triton docs print these tables **without naming the GPU**. Vector add reaches 1,684 GB/s, so it is a datacenter part with well over 1.7 TB/s (our inference); it is not a T4.

| Tutorial | Size | Triton | Torch | Naive |
|---|---|---|---|---|
| 01 vector add (GB/s) | 4,096 | 8.0 | 8.0 | – |
| 01 vector add (GB/s) | 134,217,728 | 1,684.0 | 1,684.0 | – |
| 02 softmax, M=4096 (GB/s) | N=256 | 502.6 | 706.1 | 205.7 |
| | N=1,024 | 1,015.0 | 1,076.0 | 353.8 |
| | N=4,096 | 1,391.2 | 1,315.1 | 338.7 |
| | N=8,192 | 1,435.8 | 1,148.1 | 363.4 |
| | N=12,544 | 1,457.0 | 1,393.9 | 375.2 |

On the T4, expect everything to scale down by roughly 320 / (that GPU's bandwidth). The **ratios** are what transfer: Triton ≈ 4× naive at large N, Triton vs torch.softmax within ±30% and in either direction depending on N.

Try this:
- Change `num_warps = 8` to 4 or 16 in `softmax()` and rerun the N = 4,096 row. Where does the T4 prefer it?
- Replace the persistent grid with one program per row (`num_programs = n_rows`) and compare at N = 256, where there are many cheap rows.
- In the vector add, change `BLOCK_SIZE=1024` to 128 and 4096. Small blocks mean more programs and more scheduling overhead; very large ones mean fewer programs than the 40 SMs can overlap at small sizes.